# 03 — Route Simulation & AI Suggestions
## AI City Twin Casablanca

Compare manual vs AI-suggested tram routes.

In [ ]:
import sys
sys.path.insert(0, '..')

import folium
from src.data_prep import (
    fetch_roads_offline, generate_synthetic_tram_lines,
    generate_synthetic_pois, generate_population_grid
)
from src.route_network import CityNetwork
from src.demand_model import DemandModel
from src.scenario_sim import ScenarioSimulator
from src.ai_suggester import AIRouteSuggester
from src.utils import CASABLANCA_CENTER

In [ ]:
# Build everything
roads = fetch_roads_offline()
tram = generate_synthetic_tram_lines()
pois = generate_synthetic_pois()
pop_grid = generate_population_grid()

network = CityNetwork()
network.build_from_geodataframe(roads)
network.add_tram_lines(tram)
print(network.summary())

demand_model = DemandModel(pop_grid=pop_grid, pois=pois)

In [ ]:
# Manual route simulation
sim = ScenarioSimulator(network, demand_model)
manual = sim.simulate_manual_route(33.58, -7.63, 33.55, -7.53, 'manual_1')

if manual:
    print(f'Manual route: score={manual.total_score:.0f}, '
          f'dist={manual.distance_km:.1f}km, '
          f'time={manual.travel_time_min:.0f}min')

In [ ]:
# AI suggestions
suggester = AIRouteSuggester(network, demand_model, n_clusters=8)
suggestions = suggester.suggest(top_n=3)

for s in suggestions:
    print(f"#{s['rank']} {s['candidate_id']}: "
          f"score={s['total_score']:.0f}, "
          f"dist={s['distance_km']:.1f}km, "
          f"connectivity={s['connectivity']:.2f}")

In [ ]:
# Visualize on map
m = folium.Map(location=[CASABLANCA_CENTER[0], CASABLANCA_CENTER[1]], zoom_start=12)

# Existing tram
for _, row in tram.iterrows():
    coords = [(c[1], c[0]) for c in row.geometry.coords]
    folium.PolyLine(coords, color='blue', weight=4, popup='Existing Tram').add_to(m)

# Manual route
if manual and manual.geometry:
    coords = [(c[1], c[0]) for c in manual.geometry.coords]
    folium.PolyLine(coords, color='red', weight=5, popup='Manual Route').add_to(m)

# AI suggestions
colors = ['green', 'orange', 'purple']
for i, s in enumerate(suggestions):
    coords = [(c[1], c[0]) for c in s['geometry'].coords]
    folium.PolyLine(coords, color=colors[i % 3], weight=5,
                    popup=f"AI #{s['rank']}").add_to(m)

m

In [ ]:
# Scenario comparison table
comparison = sim.compare_scenarios()
comparison